In [1]:
!pip install mysql-connector-python scikit-learn pandas

#import library
import mysql.connector
from mysql.connector import Error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import random
import pandas as pd
import math

In [2]:
# Koneksi ke database MySQL
def connect_to_db():
    return mysql.connector.connect(
        host="localhost",  # Ganti dengan host database Anda
        user="root",       # Ganti dengan user database Anda
        password="1234",       # Ganti dengan password database Anda
        database="skripsi"  # Ganti dengan nama database Anda
    )

# Mengambil data soal dari database
def fetch_questions_from_db():
    connection = connect_to_db()
    cursor = connection.cursor(dictionary=True)
    
    query = """
    SELECT id, kategori, sub_kategori, teks_soal
    FROM soal
    """
    cursor.execute(query)
    
    questions = cursor.fetchall()
    
    cursor.close()
    connection.close()
    
    return questions


In [3]:
# Menghitung cosine similarity antara soal-soal
def calculate_cosine_similarity(questions):
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([question['teks_soal'] for question in questions])
    
    cosine_sim = cosine_similarity(tfidf_matrix)
    
    return cosine_sim


In [4]:
# Mengambil soal secara merata berdasarkan kategori dan sub-kategori
def select_questions_by_category_and_subcategory(questions, total_questions=50):
    categories = list(set([q['kategori'] for q in questions]))
    subcategories = list(set([q['sub_kategori'] for q in questions]))
    
    questions_per_category = math.ceil(total_questions / len(categories))
    questions_per_subcategory = math.ceil(total_questions / len(subcategories))
    
    selected_questions = []
    
    for category in categories:
        category_questions = [q for q in questions if q['kategori'] == category]
        random.shuffle(category_questions)
        
        for subcategory in subcategories:
            subcategory_questions = [q for q in category_questions if q['sub_kategori'] == subcategory]
            if len(subcategory_questions) > 0:
                selected_questions.extend(subcategory_questions[:questions_per_subcategory])
                
        if len(selected_questions) >= total_questions:
            break
    
    if len(selected_questions) > total_questions:
        selected_questions = selected_questions[:total_questions]
    
    return selected_questions

# Mengacak soal berdasarkan cosine similarity
def shuffle_questions(questions, cosine_sim):
    shuffled_questions = []
    
    for i, question in enumerate(questions):
        sim_scores = list(enumerate(cosine_sim[i]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        similar_questions = [questions[idx] for idx, score in sim_scores if idx != i]
        
        random.shuffle(similar_questions)
        
        # Tidak mengubah id, kategori, sub_kategori, dan teks_soal
        shuffled_questions.append({
            "id": question['id'],
            "kategori": question['kategori'],
            "sub_kategori": question['sub_kategori'],
            "teks_soal": question['teks_soal']
        })
    
    return shuffled_questions


In [5]:
# Menampilkan hasil pengacakan dalam bentuk DataFrame
def display_shuffled_questions(shuffled_questions):
    df = pd.DataFrame(shuffled_questions)
    return df


In [6]:
# Main function
questions = fetch_questions_from_db()
selected_questions = select_questions_by_category_and_subcategory(questions, total_questions=50)
cosine_sim = calculate_cosine_similarity(selected_questions)
shuffled_questions = shuffle_questions(selected_questions, cosine_sim)
df = display_shuffled_questions(shuffled_questions)
df


,id,kategori,sub_kategori,teks_soal
0,3,Fisika,Mekanika,Apa rumus percepatan?
1,4,Biologi,Anatomi,Sebutkan organ utama dalam sistem pencernaan!
2,1,Matematika,Aljabar,Apa hasil dari 2 + 2?\r\na\r\nb\r\nc
3,2,Matematika,Geometri,Berapa luas lingkaran dengan radius 3?
4,5,Kimia,Organik,Apa itu hidrokarbon?
